# 模块三进阶：scVI 虚拟扰动的因果强度提升

本 Notebook 针对原 scVI 虚拟扰动的三个核心短板进行补全：
1. 阴性对照：随机选择非标志基因做同等强度扰动，建立偏移量背景分布。
2. Perturb-seq 风格验证：计算差异表达特征，与真实 CRISPR 扰动数据对比的接口。
3. scGen/CPA 路径：提供专门扰动预测模型的安装与使用模板，并展示一个简化实现。
4. 跨数据集泛化：用排名法评估子集稳定性。

In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sc.settings.verbosity = 1

## 1. 数据准备与 scVI 训练

In [ ]:
adata = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

raw_counts = adata.X.copy() if not hasattr(adata.X, 'toarray') else adata.X.toarray().copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

keep_genes = ['LYZ', 'CD14', 'MS4A1', 'IL7R', 'GNLY', 'PPBP']
for g in keep_genes:
    if g in adata.var_names:
        adata.var.loc[g, 'highly_variable'] = True

adata = adata[:, adata.var['highly_variable']].copy()

if hasattr(adata.X, 'toarray'):
    adata.X = adata.X.toarray()
adata.X = np.expm1(adata.X).round().astype(np.float32)

print(f"训练数据: {adata.n_obs} cells x {adata.n_vars} genes")

scvi.model.SCVI.setup_anndata(adata)
model = scvi.model.SCVI(adata, n_latent=10, n_hidden=128)
model.train(max_epochs=30, accelerator='cpu', plan_kwargs={'lr': 1e-3})

latent_ctrl = model.get_latent_representation(adata)
adata.obsm['X_scVI'] = latent_ctrl
print(f"对照组潜在空间形状: {latent_ctrl.shape}")

## 2. 阴性对照：随机基因扰动背景分布

In [ ]:
def perturb_gene(adata, model, gene, value=100, latent_ref=None):
    """对指定基因施加虚拟扰动并返回潜在空间偏移"""
    perturbed = adata.copy()
    if gene not in perturbed.var_names:
        return None
    idx = np.where(perturbed.var_names == gene)[0][0]
    perturbed.X[:, idx] = value
    latent_pert = model.get_latent_representation(perturbed)
    if latent_ref is None:
        latent_ref = model.get_latent_representation(adata)
    shift = np.linalg.norm(latent_pert - latent_ref, axis=1).mean()
    return shift

target_genes = ['LYZ', 'CD14', 'MS4A1']
target_shifts = {g: perturb_gene(adata, model, g, 100) for g in target_genes}

np.random.seed(42)
marker_genes = set(['LYZ', 'CD14', 'MS4A1', 'IL7R', 'GNLY', 'PPBP', 'CD3D', 'CD79A'])
candidates = [g for g in adata.var_names if g not in marker_genes]
random_genes = np.random.choice(candidates, size=20, replace=False)

random_shifts = []
for g in random_genes:
    s = perturb_gene(adata, model, g, 100)
    if s is not None:
        random_shifts.append(s)
random_shifts = np.array(random_shifts)

bg_mean = random_shifts.mean()
bg_std = random_shifts.std(ddof=1)
print(f"随机扰动背景: mean={bg_mean:.4f}, std={bg_std:.4f}")
print("\n目标基因扰动偏移与显著性:")
for g, s in target_shifts.items():
    z = (s - bg_mean) / bg_std
    p = 1 - stats.norm.cdf(z)
    print(f"  {g}: shift={s:.4f}, z={z:.2f}, p={p:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(random_shifts, bins=12, color='lightgray', edgecolor='black', alpha=0.7, label='Random genes (n=20)')
ax.axvline(bg_mean, color='black', linestyle='--', label=f'Random mean={bg_mean:.4f}')
colors = {'LYZ': 'red', 'CD14': 'blue', 'MS4A1': 'green'}
for g, s in target_shifts.items():
    ax.axvline(s, color=colors[g], linestyle='-', linewidth=2, label=f'{g}={s:.4f}')
ax.set_xlabel('Mean latent shift (L2 norm)')
ax.set_ylabel('Frequency')
ax.set_title('Negative Control: Random Gene Perturbation Background')
ax.legend(loc='upper right')
ax.grid(axis='y')
plt.tight_layout()
plt.savefig('report_images/scvi_negative_control.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 剂量响应的非线性检验

In [ ]:
doses = np.array([0, 20, 40, 60, 80, 100, 150, 200])
lyz_idx = np.where(adata.var_names == 'LYZ')[0][0]
ctrl_mean = float(adata[:, 'LYZ'].X.mean())

dose_shifts = []
for d in doses:
    perturbed = adata.copy()
    perturbed.X[:, lyz_idx] = ctrl_mean + d
    latent_pert = model.get_latent_representation(perturbed)
    dose_shifts.append(np.linalg.norm(latent_pert - latent_ctrl, axis=1).mean())
dose_shifts = np.array(dose_shifts)

x_levels = ctrl_mean + doses

lin_fit = np.polyfit(x_levels, dose_shifts, 1)
lin_pred = np.polyval(lin_fit, x_levels)
lin_r2 = 1 - np.sum((dose_shifts - lin_pred)**2) / np.sum((dose_shifts - dose_shifts.mean())**2)

quad_fit = np.polyfit(x_levels, dose_shifts, 2)
quad_pred = np.polyval(quad_fit, x_levels)
quad_r2 = 1 - np.sum((dose_shifts - quad_pred)**2) / np.sum((dose_shifts - dose_shifts.mean())**2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_levels, dose_shifts, 'o-', color='red', label='Observed', linewidth=2)
ax.plot(x_levels, lin_pred, '--', color='gray', label=f'Linear fit (R2={lin_r2:.3f})')
ax.plot(x_levels, quad_pred, ':', color='blue', label=f'Quadratic fit (R2={quad_r2:.3f})')
ax.set_xlabel('LYZ expression level')
ax.set_ylabel('Mean latent shift (L2 norm)')
ax.set_title('LYZ Dose-Response: Linearity vs Nonlinearity')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('report_images/scvi_dose_response.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"线性 R2: {lin_r2:.4f}, 二次 R2: {quad_r2:.4f}")
print(f"R2 提升: {quad_r2 - lin_r2:.4f}")

## 4. Perturb-seq 风格差异表达接口

In [ ]:
reconstructed_ctrl = model.get_normalized_expression(adata, return_mean=True)
if hasattr(reconstructed_ctrl, 'toarray'):
    reconstructed_ctrl = reconstructed_ctrl.toarray()

perturbed_lyz = adata.copy()
perturbed_lyz.X[:, lyz_idx] = 100
reconstructed_pert = model.get_normalized_expression(perturbed_lyz, return_mean=True)
if hasattr(reconstructed_pert, 'toarray'):
    reconstructed_pert = reconstructed_pert.toarray()

mean_ctrl = reconstructed_ctrl.mean(axis=0)
mean_pert = reconstructed_pert.mean(axis=0)
log2fc = np.log2((mean_pert + 1) / (mean_ctrl + 1))

de_genes = pd.DataFrame({
    'gene': adata.var_names,
    'log2FC': log2fc,
    'mean_ctrl': mean_ctrl,
    'mean_pert': mean_pert
}).sort_values('log2FC', ascending=False)

print("Top 10 上调基因（LYZ 过表达后）:")
print(de_genes.head(10).to_string(index=False))
print("\nTop 10 下调基因:")
print(de_genes.tail(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['red' if x > 0.5 else 'blue' if x < -0.5 else 'gray' for x in de_genes['log2FC']]
ax.scatter(de_genes['log2FC'], -np.log10(de_genes['mean_ctrl'] + 1), c=colors, alpha=0.6, s=20)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5)
ax.axvline(-0.5, color='blue', linestyle='--', alpha=0.5)
ax.set_xlabel('log2 Fold Change (LYZ perturbed / control)')
ax.set_ylabel('-log10(mean control expression + 1)')
ax.set_title('Virtual Perturbation DE Profile (Perturb-seq style)')
ax.grid(True)
lyz_row = de_genes[de_genes['gene'] == 'LYZ'].iloc[0]
ax.annotate('LYZ', (lyz_row['log2FC'], -np.log10(lyz_row['mean_ctrl'] + 1)),
            textcoords="offset points", xytext=(5, 5), fontsize=10, color='darkred')
plt.tight_layout()
plt.savefig('report_images/scvi_perturbseq_de.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. scGen / CPA 路径：从通用 VAE 到扰动预测模型

In [ ]:
# 安装模板（如需要可取消注释）
# !pip install scgen

# NOTE: scGen is an optional extension. The template below is not executed in this notebook.
# To use scGen, install it with: pip install scgen
scgen_template = """
import scgen
# 1. 准备带 perturbation 标签的 AnnData
#    adata.obs['condition'] = ['control'] * n_ctrl + ['LYZ_perturb'] * n_pert
#    adata.obs['cell_type'] = cell_type_labels
# 2. 训练 scGen
#    scgen.SCGEN.setup_anndata(adata, batch_key='condition', labels_key='cell_type')
#    model = scgen.SCGEN(adata)
#    model.train()
# 3. 预测扰动效应
#    pred_adata = model.predict(ctrl_key='control', stim_key='LYZ_perturb', celltype_to_predict='monocyte')
# 4. 评估：与真实 Perturb-seq 的差异表达向量对比
"""
print(scgen_template)

def learn_perturbation_direction(adata, model, gene='LYZ', high_quantile=0.9, low_quantile=0.1):
    expr = adata[:, gene].X.toarray().flatten() if hasattr(adata[:, gene].X, 'toarray') else adata[:, gene].X.flatten()
    high_mask = expr >= np.quantile(expr, high_quantile)
    low_mask = expr <= np.quantile(expr, low_quantile)
    z = model.get_latent_representation(adata)
    direction = z[high_mask].mean(axis=0) - z[low_mask].mean(axis=0)
    return direction, z

direction, z_ctrl = learn_perturbation_direction(adata, model, 'LYZ')

magnitudes = np.linspace(0, 3, 8)
direction_shifts = []
for mag in magnitudes:
    z_pert = z_ctrl + mag * direction
    direction_shifts.append(np.linalg.norm(z_pert - z_ctrl, axis=1).mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(magnitudes, direction_shifts, 'o-', color='purple')
ax.set_xlabel('Perturbation magnitude (x learned direction)')
ax.set_ylabel('Mean latent shift (L2 norm)')
ax.set_title('Perturbation-Aware Latent Direction (scGen-style)')
ax.grid(True)
plt.tight_layout()
plt.savefig('report_images/scvi_perturbation_direction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"学习到的扰动向量范数: {np.linalg.norm(direction):.4f}")

## 6. 跨数据集泛化：子采样稳定性测试（排名法）

由于两个子集训练的是独立潜在空间，**不能直接比较偏移量的绝对值**。
正确做法：在每个子集内部计算 LYZ 扰动相对于 20 个随机基因的排名/显著性，
若两次独立采样中 LYZ 均显著高于随机背景，则说明预测的方向稳定性高。

In [ ]:
np.random.seed(123)
indices = np.random.permutation(adata.n_obs)
split1 = indices[:adata.n_obs // 2]
split2 = indices[adata.n_obs // 2:]

def train_subset_scvi(adata_sub, epochs=20):
    scvi.model.SCVI.setup_anndata(adata_sub)
    m = scvi.model.SCVI(adata_sub, n_latent=10, n_hidden=128)
    m.train(max_epochs=epochs, accelerator='cpu', plan_kwargs={'lr': 1e-3})
    return m

print("训练子集 1...")
model1 = train_subset_scvi(adata[split1].copy())
print("训练子集 2...")
model2 = train_subset_scvi(adata[split2].copy())

def ranking_in_subset(adata_sub, model_sub, target='LYZ', n_random=20, seed=42):
    rng = np.random.RandomState(seed)
    marker_genes = set(['LYZ', 'CD14', 'MS4A1', 'IL7R', 'GNLY', 'PPBP', 'CD3D', 'CD79A'])
    candidates = [g for g in adata_sub.var_names if g not in marker_genes]
    random_genes = rng.choice(candidates, size=n_random, replace=False)
    
    latent_ref = model_sub.get_latent_representation(adata_sub)
    target_shift = perturb_gene(adata_sub.copy(), model_sub, target, 100, latent_ref=latent_ref)
    random_shifts = [perturb_gene(adata_sub.copy(), model_sub, g, 100, latent_ref=latent_ref) for g in random_genes]
    random_shifts = [s for s in random_shifts if s is not None]
    
    n_better = sum(1 for s in random_shifts if s >= target_shift)
    percentile = (len(random_shifts) - n_better) / len(random_shifts) * 100
    z = (target_shift - np.mean(random_shifts)) / (np.std(random_shifts, ddof=1) + 1e-9)
    return target_shift, random_shifts, percentile, z

lyz_shift1, rand1, pct1, z1 = ranking_in_subset(adata[split1].copy(), model1, 'LYZ')
lyz_shift2, rand2, pct2, z2 = ranking_in_subset(adata[split2].copy(), model2, 'LYZ')

print(f"\n子集 1: LYZ shift={lyz_shift1:.4f}, 高于 {pct1:.1f}% 随机基因, z={z1:.2f}")
print(f"子集 2: LYZ shift={lyz_shift2:.4f}, 高于 {pct2:.1f}% 随机基因, z={z2:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, rand_shifts, target_shift, pct, z, label in zip(
    axes, [rand1, rand2], [lyz_shift1, lyz_shift2], [pct1, pct2], [z1, z2], ['Split 1', 'Split 2']
):
    ax.hist(rand_shifts, bins=10, color='lightgray', edgecolor='black', alpha=0.7, label='Random genes')
    ax.axvline(target_shift, color='red', linestyle='-', linewidth=2, label=f'LYZ (z={z:.2f})')
    ax.set_xlabel('Mean latent shift (L2 norm)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{label}: LYZ > {pct:.0f}% random')
    ax.legend()
    ax.grid(axis='y')
plt.tight_layout()
plt.savefig('report_images/scvi_cross_subset.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 小节：scVI 模块升级后的能力清单

| 短板 | 原状态 | 补全内容 | 可交付物 |
|------|--------|----------|----------|
| 缺乏阴性对照 | 仅 LYZ 扰动 | 20 个随机基因背景分布 + z 检验 | scvi_negative_control.png |
| 剂量响应线性假象 | 仅描述线性 | 线性 vs 二次拟合 + R2 比较 | scvi_dose_response.png |
| 与 Perturb-seq 脱节 | 仅潜在空间偏移 | DE 特征向量 + 火山图 | scvi_perturbseq_de.png |
| 通用 VAE 因果弱 | 仅 scVI | scGen 模板 + 扰动向量学习 | scvi_perturbation_direction.png |
| 单数据集过拟合 | PBMC 3k 单一训练 | 排名法子集稳定性测试 | scvi_cross_subset.png |